# 4. Sampling Techniques — Designing Trustworthy Data Collection

**Building a Heart Disease Risk-Screening System — Notebook 4 of 12, Stage 2: Designing Trustworthy Data Collection**

Every notebook so far treated the registry as a given. This one asks where it
*comes from* — and what happens to every number computed in Notebooks 1-3 if new
patients get added to it the wrong way. Because we have the full 438-patient
registry available, we can treat it as a known population and check exactly how
much different sampling strategies would have distorted our estimates.

## The topic

**Sampling** is how a subset of a population ends up in your dataset. The
population parameter is the true value across everyone; the sample statistic is
your estimate of it. How the sample was chosen determines whether that estimate is
trustworthy — independent of how large the sample is.

## Why it matters for this system

A biased data-collection process poisons *everything downstream*: the correlations
in Notebook 5, the hypothesis tests in Notebooks 6-8, the model in Notebooks 9-12 —
all inherit whatever bias entered here, and no amount of clever modeling later
fixes a systematically unrepresentative intake process. This is the first and most
upstream place the whole system can go wrong.

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
population = pd.read_csv("../5. MLOps/2. End-to-End ML/data/heart_disease_cleaned_2.csv", index_col=0)
print(f"Treating all {len(population)} registered patients as the known population for this notebook.")

true_mean_chol = population["chol"].mean()
print(f"TRUE population mean cholesterol: {true_mean_chol:.2f} mg/dL  (the parameter)")
print("Every sampling method below pretends we don't know this exactly, and checks how")
print("close it gets using only a subset of patients.")

## The toolkit

| Method | How it works | Best for |
|---|---|---|
| **Simple random sampling (SRS)** | Every patient equally likely to be picked | The baseline; no known subgroup structure to exploit |
| **Stratified sampling** | Sample proportionally within known groups | Known subgroups (e.g. sex) that differ meaningfully |
| **Cluster sampling** | Pick whole groups, take everyone within them | Field/logistical cost makes visiting scattered individuals expensive |
| **Convenience sampling** | Whoever's easiest to reach | Rarely — it's the trap this notebook exists to demonstrate |

## How to choose

Default to SRS unless you have a specific reason not to. Switch to **stratified**
when you know a subgroup (sex, age bracket) matters and want to guarantee it's
represented rather than leaving that to chance. Switch to **cluster** when
individual-level collection is prohibitively expensive but group-level collection
is feasible (e.g. auditing whole clinics rather than scattered patients). Never
deliberately choose convenience sampling — it's included here only so its failure
mode is visible and recognizable when it happens by accident (a common real trap:
"the patients easiest to follow up with" quietly becoming your intake process).

## Applied to the registry

### Simple random sampling — the baseline

In [ ]:
def simple_random_sample(df, n, seed):
    return df.sample(n=n, random_state=seed)

srs = simple_random_sample(population, n=60, seed=1)
srs_estimate = srs["chol"].mean()
print(f"SRS estimate (n=60): {srs_estimate:.2f}   error: {abs(srs_estimate - true_mean_chol):.2f} mg/dL")

### Convenience sampling — the trap

Suppose the clinic's intake happened to skew toward whichever patients were easiest
to follow up with — say, only patients under 50 (younger patients tend to be
easier to reach for follow-up).

In [ ]:
convenience_sample = population[population["age"] < 50].sample(n=min(60, (population["age"]<50).sum()), random_state=1)
convenience_estimate = convenience_sample["chol"].mean()

print(f"Convenience sample (only under-50 patients): estimate={convenience_estimate:.2f}   "
      f"error={abs(convenience_estimate - true_mean_chol):.2f} mg/dL")
print(f"SRS error was {abs(srs_estimate - true_mean_chol):.2f} mg/dL for comparison")
print("\nCholesterol tends to rise with age, so restricting to younger patients doesn't just")
print("add noise -- it shifts the estimate in a predictable direction. More convenience-sampled")
print("data collected the same way would NOT fix this; it would just make the wrong answer")
print("more confidently wrong.")

### Stratified sampling — using known structure

`sex` is a known, meaningful subgroup in this registry. Sampling proportionally
within it guarantees representation instead of leaving it to chance.

In [ ]:
def stratified_sample(df, strata_col, n_total, seed):
    frac = n_total / len(df)
    return df.groupby(strata_col, group_keys=False).apply(
        lambda g: g.sample(frac=frac, random_state=seed), include_groups=True
    )

strat = stratified_sample(population, "sex", n_total=60, seed=1)
strat_estimate = strat["chol"].mean()
print(f"Stratified estimate (n={len(strat)}): {strat_estimate:.2f}   "
      f"error: {abs(strat_estimate - true_mean_chol):.2f} mg/dL")

print("\nSex representation, sample vs. population:")
print(pd.DataFrame({
    "sample_pct": strat["sex"].value_counts(normalize=True).round(2),
    "population_pct": population["sex"].value_counts(normalize=True).round(2),
}))

### Cluster sampling — cheaper, at a precision cost

If patients were instead grouped by, say, `restecg` result category (imagine each
category requiring a different specialist's visit to audit), cluster sampling would
pick whole categories rather than scattering across all of them.

In [ ]:
def cluster_sample(df, cluster_col, n_clusters, seed):
    chosen = df[cluster_col].drop_duplicates().sample(n=n_clusters, random_state=seed)
    return df[df[cluster_col].isin(chosen)]

cluster = cluster_sample(population, "restecg", n_clusters=1, seed=1)
cluster_estimate = cluster["chol"].mean()
print(f"Cluster sample used restecg category: {list(cluster['restecg'].unique())}")
print(f"Cluster estimate (n={len(cluster)}): {cluster_estimate:.2f}   "
      f"error: {abs(cluster_estimate - true_mean_chol):.2f} mg/dL")

### Comparing all methods across repeated samples

A single draw from each method could get lucky. Repeat 500 times and look at the
*distribution* of estimates — the honest comparison.

In [ ]:
n_trials = 500
results = {"SRS": [], "Convenience (<50)": [], "Stratified": [], "Cluster (1 restecg group)": []}

for i in range(n_trials):
    results["SRS"].append(simple_random_sample(population, 60, seed=i)["chol"].mean())
    results["Convenience (<50)"].append(convenience_estimate)  # deterministic -- that's the point
    results["Stratified"].append(stratified_sample(population, "sex", 60, seed=i)["chol"].mean())
    results["Cluster (1 restecg group)"].append(cluster_sample(population, "restecg", 1, seed=i)["chol"].mean())

summary = pd.DataFrame({
    m: {"mean_estimate": np.mean(v), "std_error": np.std(v), "avg_abs_error": np.mean(np.abs(np.array(v) - true_mean_chol))}
    for m, v in results.items()
}).T
print(f"True value: {true_mean_chol:.2f}\n")
print(summary.round(3))

The convenience sample's error is **bias** — the same wrong answer every time, no
matter how many repeats. SRS and stratified sampling average out close to the truth
(low bias); their remaining spread is genuine sampling *variability*, which more
data would reduce. This is the difference between a fixable problem (collect more)
and an unfixable one (redesign how you collect).

### Standard error and the Central Limit Theorem

The standard error of a sample mean shrinks predictably with sample size —
$SE = \sigma/\sqrt{n}$ — regardless of the underlying variable's shape.

In [ ]:
sigma = population["chol"].std()
for n in [15, 60, 150, len(population)]:
    print(f"n={n:4}  standard error = {sigma/np.sqrt(n):.3f} mg/dL")
print(f"\nGoing from n=15 to n=150 (10x the data) cuts SE by ~{1 - np.sqrt(15/150):.0%} -- roughly")
print("sqrt(10), not 10x -- diminishing returns on simply collecting more of the SAME kind of sample.")

### Confidence intervals: a usable range, not just a point estimate

In [ ]:
from scipy import stats as scipy_stats

sample = simple_random_sample(population, n=60, seed=7)
sample_mean = sample["chol"].mean()
sample_se = sample["chol"].std(ddof=1) / np.sqrt(len(sample))
ci_low, ci_high = scipy_stats.t.interval(0.95, df=len(sample)-1, loc=sample_mean, scale=sample_se)

print(f"Sample mean: {sample_mean:.2f}")
print(f"95% CI: [{ci_low:.2f}, {ci_high:.2f}]")
print(f"True population mean ({true_mean_chol:.2f}) falls inside: {ci_low <= true_mean_chol <= ci_high}")

## Systems view — what this stage hands to the next one

If Notebook 5 onward found a correlation, a significant test result, or a working
model, this notebook is what makes those findings *trustworthy* rather than
accidental — every downstream number is only as good as the sampling process that
fed it. With trustworthy data collection established, Notebook 5 moves to mapping
how the inputs relate to *each other*, the first step toward combining them into a
model.

## Try it yourself

1. Repeat the comparison using `thalach` instead of `chol` as the target variable —
   does the convenience sample's bias direction match what you'd predict from
   younger patients generally having higher max heart rates?
2. Build a stratified sample using `n_total=120` and confirm its standard error
   roughly follows the $1/\sqrt{n}$ scaling from the SE section.
3. Try cluster sampling with 2 `restecg` groups instead of 1 — does the estimate's
   error shrink noticeably, and does it still lag behind SRS/stratified of a similar
   total size?